## Timezone Conversion

**Modification Objective:** Convert event timestamps to a selected timezone.

**Motivation:** Timestamp attributes in an event log may represent time in different timezones, for example when they originate from geographically distributed systems or heterogeneous data sources. Converting a timestamp attribute to a selected timezone can establish a common temporal reference across timestamp attributes and support the correct interpretation of time-dependent information in a particular timezone. This may be relevant, for example, for determining the local date on which events occurred or for presenting event times in a timezone meaningful to end users.

**Precondition:** The event timestamps contain sufficient timezone information to determine the points in time they represent, and a target timezone is specified.

**Approach:** Convert timestamp attributes (e.g., event timestamps) to the specified target timezone while preserving the temporal information encoded by the original timestamps.

**Output:** An event log in which event timestamps are represented in the specified target timezone.


*Note that at the time of implementing this notebook, pm4py.read_xes/write_xes convert to UTC and silently drop original timezone information. Consider csv format for exporting event logs with relevant timezone information*

In [ ]:
import pandas as pd
import pm4py
import pytz

from ipywidgets import interact, Dropdown

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

In [ ]:
_common_timezones = pytz.common_timezones

timezone_widget = Dropdown(
    options=[('Select a time zone...', None)] + [(tz, tz) for tz in _common_timezones],
    value=None,
)

modified_event_log = event_log.copy()

# captured once from the freshly loaded log, before any ':tz_normalized' columns exist,
# so repeated dropdown changes always recompute from the original raw timestamps
TIMESTAMP_COLUMNS = [c for c in modified_event_log.columns if pd.api.types.is_datetime64_any_dtype(modified_event_log[c])]


@interact(target_timezone=timezone_widget)
def normalize_timezone(target_timezone):
    if target_timezone is None:
        return

    for col in TIMESTAMP_COLUMNS:
        original = modified_event_log[col]
        orignal_timezone =  original.dt.tz if original.dt.tz is not None else 'UTC'
        localized = original if original.dt.tz is not None else original.dt.tz_localize('UTC')
        modified_event_log[f'{col}_normalized_to_{target_timezone}'] = localized.dt.tz_convert(target_timezone)

    print(f'{len(TIMESTAMP_COLUMNS)} timestamp column(s) normalized from {orignal_timezone} to {target_timezone}')
    print(modified_event_log[f'{col}_normalized_to_{target_timezone}'].dtype)
    print(modified_event_log[f'{col}_normalized_to_{target_timezone}'].dt.tz)
    display(modified_event_log)